## 1. Instalación

In [ ]:
!pip install -q -U langchain langchain-classic langchain-core langchain-openai langchain-chroma langchain-huggingface sentence-transformers wikipedia gradio

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


## 2. LLM (Groq IA)

In [ ]:
import os
from google.colab import userdata
from langchain_openai import ChatOpenAI

for k in ("GROQ_API_KEY", "LLM_API_KEY"):
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        pass
os.environ.setdefault("LLM_API_KEY", os.environ.get("GROQ_API_KEY", ""))
os.environ.setdefault("LLM_BASE_URL", "https://api.groq.com/openai/v1")
os.environ.setdefault("LLM_MODEL", "openai/gpt-oss-120b")

llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ.get("GROQ_API_KEY") or os.environ["LLM_API_KEY"],
    temperature=0,
    max_retries=4,
)
print(llm.invoke("Responde solo: listo").content)

listo


## 3. Conocimiento (RAG) y memoria de largo plazo

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

emb = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

GUIAS = [
    ("Vacunas en perros", "Los cachorros inician su vacunación entre las 6 y 8 semanas, con refuerzos cada 3 a 4 semanas hasta las 16 semanas. Los adultos requieren refuerzo anual de las vacunas principales (distemper, parvovirus, adenovirus) y de rabia según la normativa local. El calendario exacto lo define un veterinario."),
    ("Vacunas en gatos", "Los gatitos inician vacunación hacia las 8 semanas, con refuerzos hasta las 16 semanas. Los adultos requieren refuerzo anual de la triple felina y de rabia según la normativa local."),
    ("Desparasitación", "Cachorros y gatitos se desparasitan desde las 2 a 3 semanas cada 2 semanas hasta los 3 meses. Los adultos, por lo general cada 3 a 6 meses (interna) y con antiparasitario externo mensual según el riesgo. Usar solo productos indicados por un veterinario y según el peso."),
    ("Alimentos peligrosos", "Chocolate, uvas, pasas, cebolla, ajo, palta, xilitol, alcohol y huesos cocidos pueden ser tóxicos o peligrosos para perros y gatos. Si hay ingestión, contactar de inmediato a un veterinario."),
    ("Alimentación general", "Dar comida balanceada formulada para su especie y etapa de vida. Cambiar de alimento gradualmente durante 7 días. Agua fresca siempre disponible. Evitar la sobrealimentación: la obesidad reduce la esperanza de vida."),
    ("Golpe de calor", "Nunca dejar mascotas dentro de un auto. Signos: jadeo intenso, babeo, encías muy rojas, debilidad. Ofrecer sombra y agua fresca (no helada) y acudir al veterinario de inmediato."),
    ("Higiene dental", "Cepillar los dientes 2 a 3 veces por semana con pasta para mascotas. El mal aliento persistente o las encías inflamadas requieren revisión veterinaria."),
    ("Ansiedad por separación", "Señales: ladridos o maullidos excesivos, destrucción de objetos, eliminar fuera de lugar al quedar solo. Ayudan las rutinas estables, el ejercicio antes de salir, juguetes de enriquecimiento y salidas graduales. Si persiste, consultar a un veterinario."),
    ("Mascotas mayores", "Desde los 7 años en perros y los 10 en gatos conviene un chequeo cada 6 meses y vigilar peso, movilidad, apetito y consumo de agua."),
    ("Adopción responsable", "Antes de adoptar: evaluar tiempo, espacio y presupuesto. Preparar cama, comedero, bebedero, identificación (chip o placa), transporte y alimento adecuado. Llevarlo a control veterinario en la primera semana, esterilizar y mantener vacunas al día. Dar tiempo de adaptación al nuevo hogar."),
    ("Ejercicio y enriquecimiento", "Los perros necesitan paseos diarios según edad y energía. Los gatos requieren juego diario, rascadores y lugares altos. El aburrimiento provoca conductas destructivas."),
]

kb = Chroma(collection_name="guias", embedding_function=emb)
kb.add_documents([Document(page_content=t, metadata={"tema": tema}) for tema, t in GUIAS])


mem = Chroma(collection_name="recuerdos", embedding_function=emb, persist_directory="./chroma_memoria")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 4. Herramientas: consulta · razonamiento · escritura

In [ ]:
import wikipedia
from langchain_core.tools import tool

wikipedia.set_lang("es")


@tool
def buscar_wikipedia(query: str) -> str:
    """Busca en Wikipedia información general sobre una raza, especie o concepto relacionado con mascotas y devuelve un resumen."""
    try:
        return wikipedia.summary(query, sentences=3)
    except wikipedia.exceptions.PageError:
        return f"No se encontró ninguna página para '{query}'."
    except wikipedia.exceptions.DisambiguationError as e:
        return f"La búsqueda '{query}' es ambigua. Opciones: {e.options[:3]}"
    except Exception as e:
        return f"Error al consultar Wikipedia: {e}"


@tool
def buscar_guia(consulta: str) -> str:
    """Consulta las guías de cuidado: vacunas, desparasitación, alimentación, higiene, comportamiento, adopción y ejercicio de perros y gatos."""
    docs = kb.similarity_search(consulta, k=3)
    return "\n\n".join(f"[{d.metadata['tema']}] {d.page_content}" for d in docs)


TOXICOS = {
    "chocolate": "TÓXICO", "cacao": "TÓXICO", "uva": "TÓXICO", "pasa": "TÓXICO", "cebolla": "TÓXICO",
    "ajo": "TÓXICO", "palta": "TÓXICO", "aguacate": "TÓXICO", "xilitol": "TÓXICO", "alcohol": "TÓXICO",
    "café": "TÓXICO", "cafe": "TÓXICO", "macadamia": "TÓXICO", "hueso cocido": "PELIGROSO",
    "leche": "PRECAUCIÓN (muchos adultos son intolerantes a la lactosa)",
}

@tool
def verificar_alimento(alimento: str, especie: str) -> str:
    """Revisa si un alimento es seguro para un perro o un gato (tóxico, peligroso o con precaución)."""
    a = alimento.lower()
    for clave, nivel in TOXICOS.items():
        if clave in a:
            return f"'{alimento}' para {especie}: {nivel}. Evitarlo; si ya lo ingirió, contactar a un veterinario."
    return f"'{alimento}' no figura en la lista de alimentos de riesgo para {especie}; darlo en poca cantidad y consultar ante alergias o dudas."


@tool
def guardar_nota(nota: str) -> str:
    """Guarda en la memoria de largo plazo un dato duradero del usuario o su mascota (nombre, especie, edad, alergias, rutina)."""
    mem.add_texts([nota])
    return f"Nota guardada: {nota}"

tools = [buscar_wikipedia, buscar_guia, verificar_alimento, guardar_nota]
DESC_TOOLS = "\n".join(f"- {t.name}: {t.description}" for t in tools)

## 5. Agente, planificación y memoria

In [ ]:
import re
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

try:
    from langchain_classic.agents import AgentExecutor, create_openai_tools_agent
except ImportError:
    from langchain.agents import AgentExecutor, create_openai_tools_agent

MARCA_TRAZA = "\n\n> 🧭"


prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres Huellas, un asistente amable que orienta a personas con perros y gatos (programa de tenencia responsable). Responde en español, de forma clara, cálida y concreta, como en un chat.
Reglas:
- Usa las herramientas cuando necesites datos (guías, Wikipedia, seguridad de alimentos); no inventes datos.
- Si el usuario cuenta algo duradero sobre su mascota (nombre, especie, edad, alergias, rutina), guárdalo con guardar_nota.
- Usa los recuerdos solo si son relevantes para la pregunta.
- No diagnostiques ni recetes medicamentos; ante síntomas preocupantes recomienda acudir a un veterinario.
- Si falta información clave (especie, edad), pídela con UNA pregunta breve.
Recuerdos del usuario:
{memoria}
Plan sugerido:
{plan}"""),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])
agente = create_openai_tools_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agente, tools=tools, max_iterations=5,
                         handle_parsing_errors=True, return_intermediate_steps=True)


def _txt(c):
    if isinstance(c, str): return c
    if isinstance(c, dict): return c.get("text", "")
    if isinstance(c, list): return " ".join(_txt(x) for x in c)
    return str(c or "")

def a_mensajes(historial, max_turnos=6):
    msgs = []
    for h in historial or []:
        if isinstance(h, dict):
            texto = _txt(h.get("content")).split(MARCA_TRAZA)[0]
            if h.get("role") == "user": msgs.append(HumanMessage(texto))
            elif h.get("role") == "assistant": msgs.append(AIMessage(texto))
        elif isinstance(h, (list, tuple)) and len(h) == 2:
            if h[0]: msgs.append(HumanMessage(_txt(h[0])))
            if h[1]: msgs.append(AIMessage(_txt(h[1]).split(MARCA_TRAZA)[0]))
    return msgs[-2 * max_turnos:]


def recuperar_memoria(consulta, k=3):
    docs = mem.similarity_search(consulta, k=k)
    return "\n".join(f"- {d.page_content}" for d in docs) or "ninguno"


def planificar(mensaje, chat_hist, recuerdos):
    reciente = "\n".join(f"{m.type}: {m.content[:150]}" for m in chat_hist[-4:]) or "ninguna"
    p = f"""Eres el planificador de un asistente de ayuda para dueños de mascotas.
Herramientas disponibles:
{DESC_TOOLS}
Responde EXACTAMENTE con este formato:
URGENCIA: alta|media|baja
PLAN:
1. ...
(máximo 3 pasos; indica la herramienta si corresponde)
Urgencia alta = posible intoxicación, golpe de calor, dificultad para respirar, convulsiones, sangrado fuerte, no puede orinar, accidente grave, temblores tras ingerir algo. Media = síntomas que ameritan revisión pronto. Baja = dudas generales, recomendaciones o charla.
Si es solo un saludo o charla, el plan es: 1. Responder directamente.
Si el usuario menciona datos duraderos de su mascota, incluye un paso con guardar_nota.
Recuerdos del usuario: {recuerdos}
Conversación reciente:
{reciente}
Mensaje: {mensaje}"""
    t = llm.invoke(p).content
    m = re.search(r"URGENCIA:\s*(alta|media|baja)", t, re.I)
    urgencia = m.group(1).lower() if m else "media"
    plan = t.split("PLAN:", 1)[1].strip() if "PLAN:" in t else "1. Responder directamente."
    return urgencia, plan

## 6. Chat con Gradio

In [ ]:
import gradio as gr

MOSTRAR_TRAZA = True
def responder(mensaje, historial):
    try:
        mensaje = _txt(mensaje)
        chat_hist = a_mensajes(historial)
        recuerdos = recuperar_memoria(mensaje)
        urgencia, plan = planificar(mensaje, chat_hist, recuerdos)

        if urgencia == "alta":
            sis = SystemMessage(content=f"""Eres Huellas, asistente de ayuda para dueños de mascotas. La situación parece una URGENCIA veterinaria.
Responde en español, breve y calmado: indica que debe llevar a su mascota a un veterinario o urgencias veterinarias de inmediato (o llamar por teléfono), da 1 o 2 acciones seguras mientras tanto (sin provocar vómito ni medicar sin indicación profesional) y qué datos tener a mano (qué ingirió o pasó, cuánto, hace cuánto, peso).
Recuerdos del usuario: {recuerdos}""")
            texto = llm.invoke([sis] + chat_hist + [HumanMessage(mensaje)]).content
            herramientas = []
        else:
            out = executor.invoke({"input": mensaje, "chat_history": chat_hist,
                                   "memoria": recuerdos, "plan": plan})
            texto = out["output"]
            herramientas = [a.tool for a, _ in out["intermediate_steps"]]

        if MOSTRAR_TRAZA:
            plan_1l = " ".join(plan.split())
            texto += (f"{MARCA_TRAZA} urgencia: {urgencia} · plan: {plan_1l} · "
                      f"herramientas: {', '.join(herramientas) or 'ninguna'}")
        return texto
    except Exception as e:
        return f"⚠️ Ocurrió un error: {str(e)[:300]}"

demo = gr.ChatInterface(
    fn=responder,
    title="🐾 Huellas — Ayuda para dueños de mascotas",
    description="Pregúntame sobre cuidados, alimentación, vacunas, adopción o comportamiento de perros y gatos. No reemplazo a un veterinario.",
    examples=[
        "Cuales son los alimentos toxicos para mi perro?",
        "A que edad es recomendable esterilizar a mi mascota?",
        "Mi perra se comió una barra de chocolate"
    ],
    cache_examples=False,
)
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3c66819fe07829b72d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3c66819fe07829b72d.gradio.live
